# Modelling — FX Activation

Thin caller of `src/run.py` (retrain) **or** a loader for a previous run, followed by a model-agnostic analysis suite. All logic lives in `src/`; the analysis works on a freshly trained model or one loaded from disk, and is **display-only** (nothing is persisted here).

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / 'configs' / '_schema.py').exists())
sys.path.insert(0, str(ROOT))
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

from configs._schema import load_config
cfg = load_config(ROOT / 'configs/fx_activation.yaml')   # validates on load

### 0 · (Optional) smoke-test the plumbing
Confirm FLAML imports and fits on throwaway data. The first import restores the `np.NaN` alias older pyspark needs but NumPy 2.0 removed (`src/run.py` does the same).

In [ ]:
from src import _compat  # restores np.NaN for pandas-on-Spark (NumPy 2); MUST precede flaml
import numpy as np, pandas as pd
from flaml import AutoML
rng = np.random.default_rng(0); n = 4000
Xs = pd.DataFrame({f'f{i}': rng.normal(size=n) for i in range(6)})
ys = (Xs['f0'] + 0.3 * rng.normal(size=n) > 0).astype(int)
m = AutoML(); m.fit(X_train=Xs, y_train=ys, task='classification', metric='roc_auc',
                    time_budget=10, estimator_list=['lgbm'], eval_method='cv',
                    n_splits=3, verbose=0)
print('best estimator:', m.best_estimator, '| plumbing OK')

### 1 · Mode — retrain or load a previous run
`retrain` runs the full `run()` pipeline and writes fresh artifacts. `load` reuses a saved run's frozen config + model (no retraining). Either way the downstream analysis is recomputed on freshly loaded OOT data, so the plots work on old models too.

In [ ]:
MODE          = 'retrain'   # 'retrain' -> full run(); 'load' -> reuse a saved run
RUN_ID        = None        # load mode: None -> latest run for this product; or pin a run_id
ARTIFACTS_DIR = 'artifacts' # where run() writes (cwd-relative, matches src/run.py)

In [ ]:
from src.run import run
from src.inference import resolve_run, load_scorer
import pathlib

if MODE == 'retrain':
    result = run(cfg, spark)
    run_dir = pathlib.Path(result['out'])
else:
    run_dir = resolve_run(cfg.product, ARTIFACTS_DIR, run_id=RUN_ID)

cfg_used, model, calibrate, model_hash = load_scorer(run_dir)   # frozen cfg + model + card offset
print('MODE:', MODE, '| run:', run_dir.name, '| model_hash:', model_hash,
      '| calibrated:', calibrate is not None)

### 2 · Load OOT and score
`score` is the deployed (calibrated) score; `score_raw` is pre-calibration. Ranking metrics are identical for both; calibration only matters for the reliability diagram.

In [ ]:
from src.dataset import load_labelled
import pandas as pd, numpy as np

X_oot, y_oot = load_labelled(spark, cfg_used, cfg_used.oot_date)
s_raw = model.predict_proba(X_oot[cfg_used.features])[:, 1]
s = calibrate(s_raw) if calibrate else s_raw
print('OOT rows:', len(X_oot), '| base rate:', round(float(y_oot.mean()), 4))

### 3 · OOT metrics (config-driven slices)
AUC + base rate always; precision/recall/lift at the `top_pct` percentiles and `top_n` absolute counts from the config (an `n` larger than the OOT population is skipped).

In [ ]:
from src.evaluate import oot_evaluate
metrics = oot_evaluate(y_oot, s, cfg_used.metric, cfg_used.top_pct, cfg_used.top_n)
metrics

### 4 · Winning model
Best estimator type, tuned hyperparameters (`best_config`), and best CV loss.

In [ ]:
import json
json.loads((run_dir / 'model_card.json').read_text())['model_selected']

### 5 · Feature importance
Permutation importance recomputed on the loaded model over OOT — model-agnostic, so it ranks numeric and raw-string categorical features on the same scale.

In [ ]:
from src import analysis
from src.importance import permutation_importance
import matplotlib.pyplot as plt

imp = permutation_importance(model, X_oot, y_oot, cfg_used.features)
analysis.importance_bar(imp); plt.show()
imp.head(15)

### 6 · Directionality — numeric features (PDP / ICE)
How the predicted score moves as a feature varies (bold = average, light = individual rows). Rising = higher feature value pushes propensity up.

In [ ]:
num = [f for f in cfg_used.features if pd.api.types.is_numeric_dtype(X_oot[f])]
top_num = [f for f in imp['feature'] if f in num][:3]
for f in top_num:
    analysis.pdp_ice(model, X_oot, cfg_used.features, f); plt.show()

### 7 · Directionality — categorical features (per level)
For each categorical feature: which levels drive the score, their target rate, lift, and direction vs the overall mean. This is the categorical counterpart to PDP.

In [ ]:
cats = [f for f in cfg_used.features if not pd.api.types.is_numeric_dtype(X_oot[f])]
for f in cats:
    tbl = analysis.categorical_direction(model, X_oot, y_oot, cfg_used.features, f)
    plt.show(); display(tbl)

### 8 · Campaign shape — cumulative gains + score distribution

In [ ]:
analysis.gains_chart(y_oot, s); plt.show()
analysis.score_distribution(s, y_oot); plt.show()

### 9 · Calibration (OOT reliability)
Predicted vs observed rate per score bin. On the diagonal = trustworthy probabilities — the direct check on the downsampling → prior-shift calibration.

In [ ]:
analysis.calibration_plot(y_oot, s); plt.show()

### 10 · (Optional) SHAP beeswarm + importance cross-check
Slow, model-agnostic SHAP over `predict_proba` for directionality, plus a check that permutation and SHAP rankings agree (disagreement often signals collinearity or leakage). Set the guard to `False` to skip.

In [ ]:
if True:   # optional; slow
    shap_imp = analysis.shap_beeswarm(model, X_oot, cfg_used.features, sample=800); plt.show()
    cross = analysis.importance_crosscheck(imp, shap_imp, other_name='shap')
    display(cross[cross['disagree']])

### 11 · Inspect the run's leakage report + scores

In [ ]:
pd.read_csv(run_dir / 'leakage_report.csv').head(10)

In [ ]:
pd.read_parquet(run_dir / 'scores.parquet').head()

### 12 · Before trusting anything, eyeball
1. Base rate & OOT AUC sane?  2. `top_pct` / `top_n` precision & recall usable for the campaign size?  3. Calibration curve near the diagonal (especially if downsampled)?  4. Gains curve well above random?  5. PDP / categorical directions make business sense; any suspected-leak feature ranking high → back to Feature Checks.  6. If downsampled, `score` is calibrated and `score_raw` is not.